# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df['is_down'] = (df['trend_direction'] == 'down').astype(int)

cols = ['impressions_90d', 'clicks_90d', 'avg_position', 'days_since_last_update', 'word_count']
print(df[cols].describe(percentiles=[0.5, 0.9, 0.99]).round(1))
print(df[cols].skew().round(1))

       impressions_90d  clicks_90d  avg_position  days_since_last_update  \
count          30000.0     30000.0       30000.0                 30000.0   
mean            5200.4        16.1          16.3                    46.1   
std            16838.0        75.1          15.2                    42.1   
min                1.0         0.0           0.0                     1.0   
50%              731.0         1.0          10.8                    20.0   
90%            12136.4        32.0          36.8                   104.0   
99%            73505.8       253.0          69.9                   106.0   
max           517715.0      4178.0         245.0                   373.0   

       word_count  
count     22301.0  
mean       3107.8  
std        1452.4  
min           8.0  
50%        2877.0  
90%        5327.0  
99%        7292.0  
max        9546.0  
impressions_90d           11.4
clicks_90d                18.3
avg_position               2.0
days_since_last_update     1.2
word_count 

Most fields have heavy right tails. For example, impressions_90d reaches 517,715 at the top while the median is [median from the table], so a few very large pages pull the averages up. Because of this I compare pages in buckets instead of trusting raw averages.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
def verdict(rates):
    r = rates.values
    if abs(r[-1] - r[0]) < 0.03:
        return "FALSE"
    if (np.diff(r) > 0).all():
        return "CONFIRMED"
    if (np.diff(r) < 0).all():
        return "OPPOSITE"
    return "MIXED"

def check(bucket):
    t = df.groupby(bucket, observed=True)['is_down'].agg(n='count', pct_declining='mean').round(3)
    print(t)
    print("verdict:", verdict(t['pct_declining']))

volume = pd.qcut(df['impressions_90d'].rank(method='first'), 4, labels=['Q1 low', 'Q2', 'Q3', 'Q4 high'])
check(volume)

words = pd.cut(df['word_count'], bins=[0, 600, 1200, 2000, np.inf], labels=['under 600', '600-1200', '1200-2000', '2000+'])
check(words)

position = pd.cut(df['avg_position'], bins=[0, 3, 10, 20, np.inf], labels=['1-3', '4-10', '11-20', '21+'])
check(position)

                    n  pct_declining
impressions_90d                     
Q1 low           7500          0.376
Q2               7500          0.605
Q3               7500          0.626
Q4 high          7500          0.562
verdict: MIXED
                n  pct_declining
word_count                      
under 600       4          0.250
600-1200     1293          0.250
1200-2000    3457          0.572
2000+       17547          0.591
verdict: MIXED
                  n  pct_declining
avg_position                      
1-3            1141          0.498
4-10          11842          0.569
11-20          7273          0.610
21+            8539          0.528
verdict: MIXED


Signal 1, search volume: verdict [word]. Declining share goes from [x] to [y] across the quarters.
Signal 2, word count: verdict [word]. Declining share goes from [x] for the shortest pages to [y] for the longest.
Signal 3, average position: verdict [word]. Declining share goes from [x] for positions 1-3 to [y] for positions 21+.
These are observed associations only. They do not show that any signal causes decline.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [4]:
stale = pd.cut(df['days_since_last_update'], bins=[-1, 90, 180, 365, np.inf], labels=['0-90', '91-180', '181-365', '365+'])
check(stale)

                            n  pct_declining
days_since_last_update                      
0-90                    20655          0.512
91-180                   9171          0.611
181-365                   169          0.467
365+                        5          0.600
verdict: MIXED


The refresh flags assume that pages not updated for a long time are more likely to decline. Verdict: [word]. The declining share goes from [x] for pages updated within 90 days to [y] for pages untouched for 365+ days. This [supports / does not support] the assumption behind the refresh flags in this starter sample.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team can use the signals marked CONFIRMED as a starting point for choosing which pages to review first, and should not rely on the signals marked FALSE or OPPOSITE. These results are directional and observational, so they support decisions but do not prove that updating a page will bring its traffic back.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.